## 📦 Step 1 — Install libraries (auto-checks, safe to re-run)

In [ ]:
import importlib, importlib.util, subprocess, sys

missing = [p for p in ("ultralytics", "easyocr") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    importlib.invalidate_caches()
print("✅ Libraries ready" if not missing else f"✅ Installed: {', '.join(missing)}")

## ⚙️ Step 2 — Imports & settings

In [ ]:
import os, re, cv2, time, base64, zipfile, urllib.request, numpy as np, pandas as pd
from collections import defaultdict, Counter
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, display

YOLO_WEIGHTS    = "yolov8s.pt"
VEHICLE_CLASSES = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
VEHICLE_CONF    = 0.30
VEHICLE_IMGSZ   = 960

PLATE_CONF      = 0.25
PLATE_IMGSZ     = 1280
MIN_PLATE_W     = 35

OCR_EVERY_N     = 2
MAX_READS       = 30
MIN_PLATE_SCORE = 0.10
MIN_VOTES       = 1
INDIAN_FORMAT   = True

IMAGE_MIN_PLATE_W = 20
IMAGE_CROP_PADS   = (0.04, 0.08, 0.15)

DISPLAY_NAME    = {"car": "Car", "motorcycle": "Bike", "bus": "Bus", "truck": "Truck"}
PLATE_THUMB_H   = 70
GALLERY_THUMB_H = 140

DEVICE          = 0 if torch.cuda.is_available() else "cpu"

print("Device:", "GPU 🚀" if DEVICE == 0 else "CPU 🐢 (works, but slow - use a GPU runtime)")

## 📤 Step 3 — Upload your photo(s) and/or video(s)

In [ ]:
INPUT_PATHS = []

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
VID_EXT = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v", ".wmv", ".flv", ".mpg", ".mpeg"}

try:
    from google.colab import files
    print("👇 Click 'Choose Files' and select your photo(s) and/or video(s)  (jpg / png / webp / mp4 / avi / mov ...)")
    uploaded = files.upload()
    INPUT_PATHS = list(uploaded.keys())
except ImportError:
    print("Not on Colab - using INPUT_PATHS =", INPUT_PATHS)


def media_kind(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in IMG_EXT:
        return "image"
    if ext in VID_EXT:
        return "video"
    if cv2.imread(path) is not None:
        return "image"
    c = cv2.VideoCapture(path); ok = c.isOpened() and c.read()[0]; c.release()
    return "video" if ok else None


def first_frame(path, kind):
    if kind == "image":
        return cv2.imread(path)
    c = cv2.VideoCapture(path); ok, f = c.read(); c.release()
    return f if ok else None


INPUTS, thumbs = [], []
for p in INPUT_PATHS:
    if not os.path.exists(p):
        print("⚠️ not found, skipped:", p); continue
    kind = media_kind(p)
    f = first_frame(p, kind) if kind else None
    if f is None:
        print("⚠️ could not read as a photo or video, skipped:", p); continue
    INPUTS.append((p, kind)); thumbs.append(f)
    h, w = f.shape[:2]
    if kind == "image":
        print(f"📷 {p} | {w}x{h}")
    else:
        c = cv2.VideoCapture(p)
        fps = c.get(cv2.CAP_PROP_FPS) or 25; n = int(c.get(cv2.CAP_PROP_FRAME_COUNT)); c.release()
        print(f"🎬 {p} | {w}x{h} | {fps:.1f} fps | {n} frames | ~{n / fps:.1f}s")

assert INPUTS, "No usable photo/video found! Upload one or set INPUT_PATHS."
n_img = sum(k == "image" for _, k in INPUTS)
print(f"\n✅ {len(INPUTS)} file(s) ready: {n_img} photo(s), {len(INPUTS) - n_img} video(s)")

show = thumbs[:6]
cols_n = min(len(show), 3); rows_n = (len(show) + cols_n - 1) // cols_n
fig, axes = plt.subplots(rows_n, cols_n, figsize=(6 * cols_n, 4 * rows_n), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for ax, f, (p, k) in zip(axes.ravel(), show, INPUTS):
    ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    ax.set_title(("photo: " if k == "image" else "video, first frame: ") + os.path.basename(p), fontsize=10)
plt.tight_layout(); plt.show()

## 🤖 Step 4 — Load models

In [ ]:
PLATE_WEIGHTS = "license_plate_detector.pt"
PLATE_URLS = [
    "https://github.com/Muhammad-Zeerak-Khan/Automatic-License-Plate-Recognition-using-YOLOv8/raw/main/license_plate_detector.pt",
    "https://huggingface.co/spaces/Harsh72AI/Vehicle-License-Plate-Detection/resolve/main/YOLOv8_best_weights.pt",
]

def weights_ok(path):
    return os.path.exists(path) and os.path.getsize(path) > 1_000_000 and zipfile.is_zipfile(path)

if not weights_ok(PLATE_WEIGHTS):
    for url in PLATE_URLS:
        try:
            print("⬇️  Downloading plate detector:", url.split("/")[2])
            urllib.request.urlretrieve(url, PLATE_WEIGHTS)
            if weights_ok(PLATE_WEIGHTS):
                break
        except Exception as e:
            print("   failed:", e)

if not weights_ok(PLATE_WEIGHTS):
    print("⚠️ Auto-download failed. Please upload your own plate-detector weights (.pt):")
    from google.colab import files
    up = files.upload()
    PLATE_WEIGHTS = list(up.keys())[0]

try:
    from ultralytics import YOLO
    import easyocr
except ModuleNotFoundError:
    import importlib, subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "easyocr"], check=True)
    importlib.invalidate_caches()
    from ultralytics import YOLO
    import easyocr

yolo_v = YOLO(YOLO_WEIGHTS)
yolo_p = YOLO(PLATE_WEIGHTS)
reader = easyocr.Reader(["en"], gpu=(DEVICE == 0))
clahe  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
print("✅ Models ready | plate detector classes:", yolo_p.names)

## 🔧 Step 5 — Helper functions

In [ ]:
ALLOW = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
INDIAN_RE = re.compile(r"^[A-Z]{2}\d{2}[A-Z]{0,3}\d{4}$")
BH_RE     = re.compile(r"^\d{2}BH\d{4}[A-Z]{2}$")
TO_LET = {"0": "O", "1": "I", "2": "Z", "5": "S", "6": "G", "8": "B", "4": "A"}
TO_DIG = {"O": "0", "Q": "0", "D": "0", "I": "1", "L": "1", "Z": "2", "S": "5", "G": "6", "B": "8", "A": "4", "T": "7"}

def clean(t):
    return re.sub(r"[^A-Z0-9]", "", t.upper())

STATE_CODES = set("AN AP AR AS BR CG CH DD DL DN GA GJ HP HR JH JK KA KL LA LD MH ML MN MP MZ NL OD OR PB PY RJ SK TN TR TS UK UP WB".split())

STATE_NAMES = {
    "AN": "Andaman and Nicobar Islands", "AP": "Andhra Pradesh", "AR": "Arunachal Pradesh",
    "AS": "Assam", "BR": "Bihar", "CG": "Chhattisgarh", "CH": "Chandigarh",
    "DD": "Daman and Diu", "DL": "Delhi", "DN": "Dadra and Nagar Haveli",
    "GA": "Goa", "GJ": "Gujarat", "HP": "Himachal Pradesh", "HR": "Haryana",
    "JH": "Jharkhand", "JK": "Jammu and Kashmir", "KA": "Karnataka", "KL": "Kerala",
    "LA": "Ladakh", "LD": "Lakshadweep", "MH": "Maharashtra", "ML": "Meghalaya",
    "MN": "Manipur", "MP": "Madhya Pradesh", "MZ": "Mizoram", "NL": "Nagaland",
    "OD": "Odisha", "OR": "Odisha", "PB": "Punjab", "PY": "Puducherry",
    "RJ": "Rajasthan", "SK": "Sikkim", "TN": "Tamil Nadu", "TR": "Tripura",
    "TS": "Telangana", "UK": "Uttarakhand", "UP": "Uttar Pradesh", "WB": "West Bengal",
}

def plate_state_name(text):
    if not text:
        return None
    if BH_RE.match(text):
        return "BH-series (All-India)"
    if INDIAN_RE.match(text):
        return STATE_NAMES.get(text[:2])
    return None

STANDARD_LAYOUTS = [(2, 1, 1, 4), (2, 1, 2, 4), (2, 1, 3, 4), (2, 2, 1, 4), (2, 2, 2, 4), (2, 2, 3, 4)]

def _try_segments(text, segments):
    result, corrections, idx = [], 0, 0
    for seg_len, seg_type in segments:
        for _ in range(seg_len):
            ch = text[idx]
            if seg_type == "L":
                if ch.isalpha():
                    result.append(ch)
                elif ch in TO_LET:
                    result.append(TO_LET[ch])
                    corrections += 1
                else:
                    return None
            else:
                if ch.isdigit():
                    result.append(ch)
                elif ch in TO_DIG:
                    result.append(TO_DIG[ch])
                    corrections += 1
                else:
                    return None
            idx += 1
    return "".join(result), corrections

def fix_indian(t):
    t = clean(t)
    if not t:
        return t
    if (INDIAN_RE.match(t) and t[:2] in STATE_CODES) or BH_RE.match(t):
        return t

    MAX_TOTAL_EDITS = 3
    best, best_key = None, None

    for lead in range(0, 3):
        for trail in range(0, 3):
            core = t[lead:len(t) - trail] if trail else t[lead:]
            if len(core) < 8 or (lead + trail) > MAX_TOTAL_EDITS:
                continue

            for state_len, rto_len, series_len, num_len in STANDARD_LAYOUTS:
                segs = [(state_len, "L"), (rto_len, "D"), (series_len, "L"), (num_len, "D")]
                if len(core) != sum(s for s, _ in segs):
                    continue
                out = _try_segments(core, segs)
                if out is None:
                    continue
                corrected, ncorr = out
                total_edits = lead + trail + ncorr
                if total_edits > MAX_TOTAL_EDITS or not INDIAN_RE.match(corrected):
                    continue
                key = (0 if corrected[:2] in STATE_CODES else 1, total_edits)
                if best_key is None or key < best_key:
                    best, best_key = corrected, key

            if len(core) == 10:
                out = _try_segments(core, [(2, "D"), (2, "L"), (4, "D"), (2, "L")])
                if out is not None:
                    corrected, ncorr = out
                    total_edits = lead + trail + ncorr
                    if total_edits <= MAX_TOTAL_EDITS and corrected[2:4] == "BH":
                        key = (0, total_edits)
                        if best_key is None or key < best_key:
                            best, best_key = corrected, key

    return best if best is not None else t

def score_text(t, conf):
    t = clean(t)
    if not 4 <= len(t) <= 12:
        return 0.0
    if sum(c.isdigit() for c in t) < 2:
        return 0.0
    if INDIAN_FORMAT:
        if INDIAN_RE.match(t) or BH_RE.match(t):
            bonus = 1.0
        elif 6 <= len(t) <= 10:
            bonus = 0.6
        else:
            bonus = 0.4
    else:
        bonus = 1.0 if 5 <= len(t) <= 10 else 0.5
    return conf * bonus

def ocr_variants(crop):
    h, w = crop.shape[:2]
    scale = min(max(1.0, 320.0 / max(w, 1)), 4.0)
    big = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC) if scale > 1 else crop
    gray = cv2.cvtColor(big, cv2.COLOR_BGR2GRAY)
    g_clahe = clahe.apply(gray)
    _, g_bin = cv2.threshold(cv2.GaussianBlur(gray, (3, 3), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return [big, g_clahe, g_bin]

def read_pieces(img):
    try:
        res = reader.readtext(img, allowlist=ALLOW, detail=1, paragraph=False, text_threshold=0.5, low_text=0.3)
    except Exception:
        return []
    if not res:
        return []
    hh = img.shape[0]
    items = []
    for box, txt, cf in res:
        yc = (box[0][1] + box[2][1]) / 2.0
        xc = (box[0][0] + box[2][0]) / 2.0
        items.append((yc, xc, txt.upper(), float(cf)))
    ys = [i[0] for i in items]
    if max(ys) - min(ys) > 0.30 * hh:
        mid = (max(ys) + min(ys)) / 2.0
        items.sort(key=lambda i: (i[0] > mid, i[1]))
    else:
        items.sort(key=lambda i: i[1])
    return [(i[2], i[3]) for i in items]

def ocr_plate(crop):
    best_text, best_score, best_img = "", 0.0, None
    raw_best_text, raw_best_conf = "", 0.0

    for img in ocr_variants(crop):
        pieces = read_pieces(img)
        if not pieces:
            continue
        joined_text = "".join(p[0] for p in pieces)
        joined_conf = float(np.mean([p[1] for p in pieces]))

        for text, conf in [(joined_text, joined_conf)] + pieces:
            options = [text] + ([fix_indian(text)] if INDIAN_FORMAT else [])
            for t in options:
                s = score_text(t, conf)
                if s > best_score:
                    best_score, best_text, best_img = s, clean(t), img

        if joined_conf > raw_best_conf:
            raw_best_conf, raw_best_text = joined_conf, joined_text

    return best_text, best_score, clean(raw_best_text), best_img

def enhance_for_display(img, target_h=GALLERY_THUMB_H):
    if img is None or img.size == 0:
        return img
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    h, w = img.shape[:2]
    if h == 0 or w == 0:
        return img
    scale = target_h / float(h)
    if scale > 1.0:
        img = cv2.resize(img, (max(1, int(w * scale)), target_h), interpolation=cv2.INTER_CUBIC)
    blur = cv2.GaussianBlur(img, (0, 0), 1.2)
    return cv2.addWeighted(img, 1.5, blur, -0.5, 0)

def paste_thumbnail(vis, thumb, x, y):
    if thumb is None or thumb.size == 0:
        return
    H, W = vis.shape[:2]
    th, tw = thumb.shape[:2]
    if tw >= W or th >= H:
        return
    x = max(2, min(x, W - tw - 2))
    y = max(2, min(y, H - th - 2))
    cv2.rectangle(vis, (x - 3, y - 3), (x + tw + 3, y + th + 3), (255, 255, 255), -1)
    cv2.rectangle(vis, (x - 3, y - 3), (x + tw + 3, y + th + 3), (30, 30, 30), 2)
    vis[y:y + th, x:x + tw] = thumb

def crop_box(frame, box, pad=0.08):
    x1, y1, x2, y2 = [int(v) for v in box]
    pw, ph = int((x2 - x1) * pad), int((y2 - y1) * pad)
    x1, y1 = max(0, x1 - pw), max(0, y1 - ph)
    x2, y2 = min(frame.shape[1], x2 + pw), min(frame.shape[0], y2 + ph)
    return frame[y1:y2, x1:x2]

def iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0

class PlateTracker:
    def __init__(self, max_gap=20):
        self.last, self.next_id, self.max_gap = {}, 1, max_gap

    def update(self, boxes, fi):
        ids, used = [], set()
        for b in boxes:
            best, best_s = None, 0.0
            size = max(b[2] - b[0], b[3] - b[1], 1)
            cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2

            for tid, (pb, pf) in self.last.items():
                if tid in used or fi - pf > self.max_gap:
                    continue
                px, py = (pb[0] + pb[2]) / 2, (pb[1] + pb[3]) / 2
                d = ((cx - px) ** 2 + (cy - py) ** 2) ** 0.5
                s = iou(b, pb)
                if s < 0.05 and d > 2.0 * size:
                    continue
                s2 = s + max(0.0, 1 - d / (2.0 * size)) * 0.5
                if s2 > best_s:
                    best, best_s = tid, s2

            if best is None:
                best = self.next_id
                self.next_id += 1

            used.add(best)
            self.last[best] = (b, fi)
            ids.append(best)

        return ids

def put_label(img, text, org, color, scale=0.6, thick=2):
    (tw, th), bl = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thick)
    x, y = org
    cv2.rectangle(img, (x, y - th - bl - 4), (x + tw + 6, y + 2), color, -1)
    cv2.putText(img, text, (x + 3, y - bl), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), thick, cv2.LINE_AA)

print("✅ Helpers ready")

## 🧪 Step 6 — Quick test on your photo(s) / 6 sample video frames (do this BEFORE the full run)

In [ ]:
samples = []
per_video = max(1, 6 // len(INPUTS))

for p, kind in INPUTS:
    name = os.path.basename(p)
    if kind == "image":
        samples.append((name, cv2.imread(p)))
    else:
        cap = cv2.VideoCapture(p)
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        for i in np.linspace(0, max(n - 1, 0), per_video).astype(int):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
            ok, f = cap.read()
            if ok:
                samples.append((f"{name} | frame {i}", f))
        cap.release()

if len(samples) > 6:
    print(f"Showing the first 6 of {len(samples)} samples")

samples = samples[:6]
assert samples, "Could not read any frame from the uploaded files."

cols_n = min(len(samples), 3)
rows_n = (len(samples) + cols_n - 1) // cols_n
fig, axes = plt.subplots(rows_n, cols_n, figsize=(6 * cols_n, 4.5 * rows_n), squeeze=False)

for ax in axes.ravel():
    ax.axis("off")

for ax, (title, f) in zip(axes.ravel(), samples):
    pr = yolo_p.predict(f, conf=PLATE_CONF, imgsz=PLATE_IMGSZ, device=DEVICE, verbose=False)[0]
    vis, reads = f.copy(), []
    lw = max(3, int(max(f.shape[:2]) / 400))

    for b, cf in zip(pr.boxes.xyxy.cpu().numpy(), pr.boxes.conf.cpu().numpy()):
        x1, y1, x2, y2 = [int(v) for v in b]
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 255), lw)
        text, score, raw, _ = ocr_plate(crop_box(f, b))
        reads.append(text or f"?({raw})")

    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.axis("off")
    ax.set_title(f"{title}\n" + (", ".join(reads) if reads else "no plate found"), fontsize=10)

plt.tight_layout()
plt.show()

## ⚙️ Step 7 — Processing engine

In [ ]:
COLORS = {"car": (200, 120, 0), "motorcycle": (0, 140, 255), "bus": (0, 160, 0), "truck": (150, 0, 150)}
_used_out = set()

def new_state():
    return {
        "ptracker": PlateTracker(),
        "plates": defaultdict(lambda: {"counts": Counter(), "scores": defaultdict(float), "crops": {}, "sample": None,
                                       "first": None, "last": None, "vtypes": Counter(), "vids": Counter(),
                                       "n_reads": 0, "raw": Counter()}),
        "stats": Counter(), "raw_log": [], "previews": [],
    }

def best_text(tr):
    return max(tr["scores"], key=lambda t: tr["scores"][t]) if tr["scores"] else None

def unique_out(path, suffix):
    stem = re.sub(r"[^A-Za-z0-9_-]+", "_", os.path.splitext(os.path.basename(path))[0]) or "file"
    out, k = f"annotated_{stem}{suffix}", 2
    while out in _used_out:
        out = f"annotated_{stem}_{k}{suffix}"
        k += 1
    _used_out.add(out)
    return out

def process_frame(frame, fi, S, tracker_model=None):
    is_video = tracker_model is not None
    plates, stats, raw_log = S["plates"], S["stats"], S["raw_log"]
    vis = frame.copy()
    sc = 1.0 if is_video else max(1.0, max(frame.shape[:2]) / 1000.0)
    th = max(1, int(round(2 * sc)))
    stats["frames"] += 1

    vopts = dict(classes=list(VEHICLE_CLASSES), conf=VEHICLE_CONF, imgsz=VEHICLE_IMGSZ, device=DEVICE, verbose=False)
    if is_video:
        vres = tracker_model.track(frame, persist=True, tracker="bytetrack.yaml", **vopts)[0]
    else:
        vres = yolo_v.predict(frame, **vopts)[0]

    vehicles = []
    if vres.boxes is not None and len(vres.boxes):
        vb = vres.boxes.xyxy.cpu().numpy().astype(int)
        vc = vres.boxes.cls.cpu().numpy().astype(int)
        if is_video:
            vi = vres.boxes.id.cpu().numpy().astype(int) if vres.boxes.id is not None else np.zeros(len(vb), int)
        else:
            vi = np.arange(1, len(vb) + 1)
        vehicles = [(vb[k], int(vi[k]), VEHICLE_CLASSES.get(int(vc[k]), "vehicle")) for k in range(len(vb))]
        stats["frames_with_vehicle"] += 1
        stats["vehicles"] += len(vehicles)

    pres = yolo_p.predict(frame, conf=PLATE_CONF, imgsz=PLATE_IMGSZ, device=DEVICE, verbose=False)[0]
    pboxes = pres.boxes.xyxy.cpu().numpy() if pres.boxes is not None and len(pres.boxes) else np.zeros((0, 4))
    pids = S["ptracker"].update([tuple(b) for b in pboxes], fi)

    if len(pboxes):
        stats["frames_with_plate"] += 1

    veh_plate = {}

    for pb, pid in zip(pboxes, pids):
        x1, y1, x2, y2 = [int(v) for v in pb]
        tr = plates[pid]
        tr["first"] = fi if tr["first"] is None else tr["first"]
        tr["last"] = fi
        stats["plate_boxes"] += 1

        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        owner = None

        for k, (vbox, vid, vtype) in enumerate(vehicles):
            if vbox[0] <= cx <= vbox[2] and vbox[1] <= cy <= vbox[3]:
                area = (vbox[2] - vbox[0]) * (vbox[3] - vbox[1])
                if owner is None or area < owner[1]:
                    owner = (k, area)

        if owner is not None:
            tr["vtypes"][vehicles[owner[0]][2]] += 1
            tr["vids"][vehicles[owner[0]][1]] += 1

        crop = crop_box(frame, pb)

        if tr["sample"] is None or crop.size > tr["sample"].size:
            tr["sample"] = crop.copy()

        if is_video:
            do_ocr = fi % OCR_EVERY_N == 0 and (x2 - x1) >= MIN_PLATE_W and tr["n_reads"] < MAX_READS
            pads = (0.08,)
        else:
            do_ocr = (x2 - x1) >= IMAGE_MIN_PLATE_W
            pads = IMAGE_CROP_PADS

        if do_ocr and crop.size:
            for pad in pads:
                c = crop_box(frame, pb, pad)
                if not c.size:
                    continue

                tr["n_reads"] += 1
                stats["ocr_calls"] += 1
                text, score, raw, best_img = ocr_plate(c)

                if raw:
                    tr["raw"][raw] += 1

                if len(raw_log) < 400:
                    raw_log.append({
                        "frame": fi,
                        "plate_id": pid,
                        "raw_ocr": raw,
                        "accepted_text": text,
                        "score": round(score, 2)
                    })

                if text and score >= MIN_PLATE_SCORE:
                    stats["accepted_reads"] += 1
                    tr["counts"][text] += 1
                    tr["scores"][text] += score

                    if text not in tr["crops"] or score > tr["crops"][text][0]:
                        tr["crops"][text] = (
                            score,
                            enhance_for_display(best_img if best_img is not None else c)
                        )

        shown = best_text(tr)

        if owner is not None and shown:
            veh_plate[owner[0]] = shown

        cv2.rectangle(
            vis,
            (x1, y1),
            (x2, y2),
            (0, 230, 255),
            max(3, int(round(3 * sc)))
        )

        if shown:
            state = plate_state_name(shown)
            shown_display = f"{shown} ({state})" if state else shown

            put_label(
                vis,
                shown_display,
                (x1, max(y1 - 4, int(24 * sc))),
                (0, 120, 200),
                scale=0.8 * sc,
                thick=th
            )

            if shown in tr["crops"]:
                clear_img = tr["crops"][shown][1]

                if clear_img is not None and clear_img.size:
                    ch, cw = clear_img.shape[:2]
                    thumb_h = max(28, int(PLATE_THUMB_H * sc))
                    thumb_w = max(1, int(cw * (thumb_h / float(ch))))
                    thumb = cv2.resize(
                        clear_img,
                        (thumb_w, thumb_h),
                        interpolation=cv2.INTER_LINEAR
                    )

                    tx = x1
                    ty = y2 + int(10 * sc)

                    if ty + thumb_h + 4 > vis.shape[0]:
                        ty = max(0, y1 - thumb_h - int(10 * sc))

                    paste_thumbnail(vis, thumb, tx, ty)

    for k, (vbox, vid, vtype) in enumerate(vehicles):
        col = COLORS.get(vtype, (80, 80, 80))

        cv2.rectangle(
            vis,
            (int(vbox[0]), int(vbox[1])),
            (int(vbox[2]), int(vbox[3])),
            col,
            max(2, int(round(2 * sc)))
        )

        vname = DISPLAY_NAME.get(vtype, vtype.title())
        plate_txt = veh_plate.get(k)

        if plate_txt:
            state = plate_state_name(plate_txt)
            plate_txt = f"{plate_txt} ({state})" if state else plate_txt

        label = f"#{vid} {vname}" + (f" | {plate_txt}" if plate_txt else "")

        put_label(
            vis,
            label,
            (int(vbox[0]), max(int(vbox[1]), int(22 * sc))),
            col,
            scale=0.6 * sc,
            thick=th
        )

    if is_video and len(pboxes) and fi % 15 == 0 and len(S["previews"]) < 6:
        S["previews"].append(vis.copy())

    return vis

def run_image(path):
    frame = cv2.imread(path)

    if frame is None:
        raise ValueError("OpenCV could not read this image")

    S = new_state()
    vis = process_frame(frame, 0, S)
    out = unique_out(path, ".jpg")

    cv2.imwrite(
        out,
        vis,
        [cv2.IMWRITE_JPEG_QUALITY, 95]
    )

    return {
        "name": os.path.basename(path),
        "kind": "image",
        "state": S,
        "out": out,
        "previews": [vis],
        "fps": None
    }

def run_video(path):
    cap = cv2.VideoCapture(path)

    if not cap.isOpened():
        raise ValueError("OpenCV could not open this video")

    vw, vh = int(cap.get(3)), int(cap.get(4))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    raw_out = unique_out(path, "_raw.mp4")
    writer = cv2.VideoWriter(
        raw_out,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (vw, vh)
    )

    vmodel = YOLO(YOLO_WEIGHTS)
    S = new_state()
    fi = 0

    with tqdm(total=n if n > 0 else None, desc=os.path.basename(path)) as bar:
        while True:
            ok, frame = cap.read()

            if not ok:
                break

            writer.write(process_frame(frame, fi, S, vmodel))
            fi += 1
            bar.update(1)

    cap.release()
    writer.release()

    return {
        "name": os.path.basename(path),
        "kind": "video",
        "state": S,
        "out": raw_out,
        "previews": S["previews"],
        "fps": fps
    }

print("✅ Engine ready (photo + video)")

## ▶️ Step 8 — Run on everything you uploaded

In [ ]:
import traceback

RESULTS = []
_used_out.clear()
t0 = time.time()

def show_diagnostics(res):
    st = res["state"]["stats"]
    print("🩺 DIAGNOSTICS")

    if res["kind"] == "video":
        print(f"   frames processed        : {st['frames']}")
        print(f"   frames with a vehicle   : {st['frames_with_vehicle']}")
        print(f"   frames with a plate box : {st['frames_with_plate']}   (plate boxes total: {st['plate_boxes']})")
    else:
        print(f"   vehicles found          : {st['vehicles']}")
        print(f"   plate boxes found       : {st['plate_boxes']}")

    print(f"   OCR attempts            : {st['ocr_calls']}   accepted reads: {st['accepted_reads']}")

    if st["plate_boxes"] == 0:
        print("\n❌ The plate detector found NO plates. Try: PLATE_CONF = 0.15, a higher-resolution / closer shot,")
        print("   or upload a different plate-detector .pt in Step 4.")
    elif st["accepted_reads"] == 0:
        print("\n⚠️ Plates were found but OCR could not read them. Plates are probably too small/blurry (< ~60 px wide).")
        print("   Look at the crops in Step 9 to see what the OCR sees; try a sharper / closer photo or video.")

for p, kind in INPUTS:
    print(f"\n{'📷 PHOTO' if kind == 'image' else '🎬 VIDEO'}: {os.path.basename(p)}")

    try:
        res = run_image(p) if kind == "image" else run_video(p)
    except Exception:
        print(f"❌ failed on {p}:")
        traceback.print_exc()
        continue

    RESULTS.append(res)
    show_diagnostics(res)

    if res["kind"] == "image":
        plt.figure(figsize=(12, 8))
        plt.imshow(cv2.cvtColor(res["previews"][0], cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.title(res["name"])
        plt.show()

    elif res["previews"]:
        pv = res["previews"][:6]
        fig, axes = plt.subplots(2, 3, figsize=(18, 8))

        for ax in axes.ravel():
            ax.axis("off")

        for ax, f in zip(axes.ravel(), pv):
            ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))

        plt.tight_layout()
        plt.show()

print(f"\n✅ All done in {time.time() - t0:.0f}s  ({len(RESULTS)}/{len(INPUTS)} file(s) processed)")

## 📋 Step 9 — Final number plates + plate crops + OCR debug log

In [ ]:
rows = []

for order, res in enumerate(RESULTS):
    is_img = res["kind"] == "image"

    for pid, tr in res["state"]["plates"].items():
        text = best_text(tr)

        if not text or tr["counts"][text] < MIN_VOTES:
            continue

        rows.append({
            "_o": order,
            "Source": res["name"],
            "Type": "photo" if is_img else "video",
            "Vehicle Type": DISPLAY_NAME.get(
                tr["vtypes"].most_common(1)[0][0], "Vehicle"
            ) if tr["vtypes"] else "Vehicle",
            "Vehicle ID": tr["vids"].most_common(1)[0][0] if tr["vids"] else "-",
            "Number Plate": text,
            "State": plate_state_name(text) or "-",
            "Avg Confidence": round(
                tr["scores"][text] / tr["counts"][text], 2
            ),
            "Times Read": tr["counts"][text],
            "First Seen (s)": np.nan if is_img else round(
                tr["first"] / res["fps"], 1
            ),
            "Last Seen (s)": np.nan if is_img else round(
                tr["last"] / res["fps"], 1
            ),
        })

cols = [
    "Source", "Type", "Vehicle Type", "Vehicle ID", "Number Plate",
    "State", "Avg Confidence", "Times Read", "First Seen (s)", "Last Seen (s)"
]

df = pd.DataFrame(rows, columns=["_o"] + cols)

if not df.empty:
    df = (
        df.sort_values("Times Read", ascending=False)
          .drop_duplicates(["Source", "Number Plate"])
          .sort_values(["_o", "First Seen (s)"], kind="stable")
          .reset_index(drop=True)
    )

df = df[cols]

for c in ["First Seen (s)", "Last Seen (s)"]:
    df[c] = df[c].astype(object).where(df[c].notna(), "-")

df.to_csv("detected_plates.csv", index=False)

print(f"🚦 {len(df)} unique number plate(s) detected in {len(RESULTS)} file(s)")
display(df)

items = [
    (res["name"], tr)
    for res in RESULTS
    for tr in res["state"]["plates"].values()
    if tr["sample"] is not None
][:12]

if items:
    cols_n = min(len(items), 4)
    rows_n = (len(items) + cols_n - 1) // cols_n

    fig, axes = plt.subplots(
        rows_n,
        cols_n,
        figsize=(4.5 * cols_n, 2.8 * rows_n),
        squeeze=False
    )

    for ax in axes.ravel():
        ax.axis("off")

    for ax, (name, tr) in zip(axes.ravel(), items):
        t = best_text(tr)

        img = (
            tr["crops"][t][1]
            if t and t in tr["crops"]
            else enhance_for_display(tr["sample"])
        )

        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

        raw = tr["raw"].most_common(1)[0][0] if tr["raw"] else "-"
        vtype = (
            DISPLAY_NAME.get(
                tr["vtypes"].most_common(1)[0][0],
                "Vehicle"
            )
            if tr["vtypes"]
            else "Vehicle"
        )

        state = plate_state_name(t) if t else None

        read_line = f"read: {t or 'NONE'}" + (
            f" ({state})" if state else ""
        )

        ax.set_title(
            f"{name[:28]} | {vtype}\n"
            f"{read_line}\n"
            f"raw OCR: {raw}",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

print("\n🔍 First OCR attempts (debug log):")

dbg = [
    dict(r, source=res["name"])
    for res in RESULTS
    for r in res["state"]["raw_log"]
]

display(pd.DataFrame(dbg).head(25))

## 🎥 Step 10 — Watch & download results

In [ ]:
import subprocess

outputs = []

for res in RESULTS:
    if res["kind"] == "image":
        outputs.append(res["out"])
        continue

    raw = res["out"]
    final = raw.replace("_raw.mp4", ".mp4")

    try:
        r = subprocess.run(
            [
                "ffmpeg",
                "-y",
                "-loglevel",
                "error",
                "-i",
                raw,
                "-vcodec",
                "libx264",
                "-pix_fmt",
                "yuv420p",
                final
            ],
            capture_output=True,
            text=True
        )

        if r.returncode != 0 or not os.path.exists(final):
            raise RuntimeError(r.stderr.strip() or "ffmpeg failed")

    except Exception as e:
        print(
            f"⚠️ ffmpeg re-encode failed for {res['name']} ({e}) "
            f"- using the raw file (may not play in the browser)"
        )
        final = raw

    outputs.append(final)

    size_mb = os.path.getsize(final) / 1e6
    print(f"🎬 {res['name']} -> {final} ({size_mb:.1f} MB)")

    if size_mb < 25:
        b64 = base64.b64encode(open(final, "rb").read()).decode()
        display(
            HTML(
                f'<video width="720" controls>'
                f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
                f'</video>'
            )
        )
    else:
        print(
            f"   Video is {size_mb:.0f} MB - too big to embed, "
            f"use the download below."
        )

outputs.append("detected_plates.csv")

try:
    from google.colab import files

    if len(outputs) > 2:
        with zipfile.ZipFile("anpr_results.zip", "w") as z:
            for f in outputs:
                z.write(f)

        files.download("anpr_results.zip")
    else:
        for f in outputs:
            files.download(f)

except ImportError:
    print("Saved locally:", ", ".join(outputs))